# LeetCode 382 — Linked List Random Node bằng Python

Link đề: https://leetcode.com/problems/linked-list-random-node/ · Độ khó: **Medium** · Chủ đề: Linked List, Math, Reservoir Sampling, Randomized

Skeleton LeetCode:

```python
# Definition for singly-linked list.
# class ListNode:
#     def __init__(self, val=0, next=None):
#         self.val = val
#         self.next = next
class Solution:

    def __init__(self, head: Optional[ListNode]):
        ...

    def getRandom(self) -> int:
        ...


# Your Solution object will be instantiated and called as such:
# obj = Solution(head)
# param_1 = obj.getRandom()
```

Ghi chú: bài **ngẫu nhiên** nên LeetCode không so đáp án cứng. Trong notebook ta tự dựng `ListNode`, bộ chạy chuỗi lệnh, và test bằng **tính chất**, **thống kê có seed**, và cả một phép **kiểm chứng chính xác** (đếm toàn bộ khả năng của các lần tung đồng xu) để chứng minh xác suất đều.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: cho danh sách liên kết đơn, mỗi lần `getRandom()` phải trả giá trị của **một nút được chọn với xác suất bằng nhau**; câu hỏi mở rộng: danh sách **rất dài, không biết độ dài** và **không dùng thêm bộ nhớ**.
2. Nắm các công cụ Python: tự dựng `ListNode`, duyệt `while node:`, `random.randrange`, `random.random`, `random.seed`, `fractions.Fraction` để tính xác suất **chính xác**.
3. Đi từ **chép ra mảng** → **đếm độ dài mỗi lần rồi nhảy tới nút ngẫu nhiên** → **Reservoir Sampling (một lượt, `O(1)` bộ nhớ)** → **gán khóa ngẫu nhiên và lấy nhỏ nhất**.
4. Hiểu mấu chốt của Reservoir Sampling: ở nút thứ `i`, **thay đáp án bằng nút đó với xác suất `1/i`** thì kết thúc, mọi nút đều có xác suất `1/n`.
5. Tránh các lỗi kinh điển: thay với xác suất `1/2` thay vì `1/i`, sai biên của `randrange`/`randint`, nhầm "giá trị" với "nút" khi có giá trị trùng, và test ngẫu nhiên bằng so sánh cứng.

## 2. Đề bài & ví dụ

Cho một danh sách liên kết đơn. Hãy cài lớp `Solution`:

- `Solution(head)`: khởi tạo với nút đầu `head`.
- `getRandom()`: chọn **ngẫu nhiên một nút** và trả về giá trị của nó. **Mọi nút** phải có xác suất được chọn **như nhau**.

**Ví dụ:** `Solution([1, 2, 3])`, gọi `getRandom()` 5 lần có thể ra `[1, 3, 2, 2, 3]` — bất kỳ dãy nào chỉ gồm `1, 2, 3` đều hợp lệ, miễn mỗi giá trị có xác suất `1/3`.

**Ràng buộc:** số nút trong `[1, 10^4]`; `-10^4 <= Node.val <= 10^4`; tối đa `10^4` lần gọi `getRandom`.

**Follow up:** nếu danh sách cực lớn và **không biết độ dài** thì sao? Có làm được mà **không dùng thêm bộ nhớ** không?

Lưu ý: giá trị có thể **trùng nhau**; xác suất tính theo **nút**, nên giá trị xuất hiện nhiều lần sẽ có xác suất lớn hơn.

In [1]:
import random
import time
from collections import Counter
from fractions import Fraction
from itertools import product
from typing import List, Optional


class ListNode:
    def __init__(self, val=0, next=None):
        self.val = val
        self.next = next


def build_list(vals):
    """Dựng danh sách liên kết từ list Python; trả về nút đầu (None nếu rỗng)."""
    dummy = ListNode()
    cur = dummy
    for v in vals:
        cur.next = ListNode(v)
        cur = cur.next
    return dummy.next


def to_list(head):
    out = []
    while head:
        out.append(head.val)
        head = head.next
    return out


def run_ops(cls, ops, args):
    """Mô phỏng LeetCode: ops[0] là hàm tạo (tham số đầu là list -> đổi thành ListNode)."""
    obj, out = None, []
    for i, (op, a) in enumerate(zip(ops, args)):
        if i == 0:
            obj = cls(build_list(a[0]))
            out.append(None)
        else:
            out.append(getattr(obj, op)(*a))
    return out


ops = ["Solution", "getRandom", "getRandom", "getRandom", "getRandom", "getRandom"]
args = [[[1, 2, 3]], [], [], [], [], []]
head = build_list([1, 2, 3])
print(to_list(head))

[1, 2, 3]


## 3. Kiến thức nền

### 3.1. Danh sách liên kết đơn và cách duyệt

Mỗi nút có `val` và `next` (nút kế, `None` ở cuối). Ta **không truy cập theo chỉ số được**: muốn tới nút thứ `k` phải đi qua `k` nút. Duyệt chuẩn: `while node: ...; node = node.next`.

In [2]:
head = build_list([10, 20, 30, 40])
node = head
steps = 0
while node:
    print("nút", steps, "có val =", node.val)
    node = node.next
    steps += 1
print("độ dài =", steps)

# đi tới nút thứ 2 (đếm từ 0): phải bước 2 lần
node = head
for _ in range(2):
    node = node.next
print(node.val)    # 30

nút 0 có val = 10
nút 1 có val = 20
nút 2 có val = 30
nút 3 có val = 40
độ dài = 4
30


### 3.2. Module `random`

- `random.randrange(n)`: số nguyên đều trong `[0, n)` (không bao giờ ra `n`).
- `random.randint(a, b)`: đều trong `[a, b]` — **bao gồm cả `b`**.
- `random.random()`: số thực đều trong `[0, 1)`.
- `random.choice(seq)`: phần tử ngẫu nhiên của dãy có chỉ số.
- `random.seed(x)`: cố định dãy ngẫu nhiên, để test **lặp lại được**.

In [3]:
random.seed(42)
print(random.randrange(3), random.randint(1, 3), round(random.random(), 4), random.choice("abc"))
random.seed(42)
print(random.randrange(3), random.randint(1, 3), round(random.random(), 4), random.choice("abc"))   # y hệt

# randrange(1) luôn là 0 -> "xác suất 1/1 = 100%"
print({random.randrange(1) for _ in range(100)})

2 1 0.025 b
2 1 0.025 b
{0}


### 3.3. "Xác suất `1/i`" bằng `randrange`

Muốn một sự kiện xảy ra với xác suất `1/i`: `random.randrange(i) == 0`. Có `i` kết quả đều nhau (`0..i-1`), chỉ một trong chúng là `0`. Kiểm tra bằng đếm:

In [4]:
random.seed(1)
for i in (1, 2, 3, 5):
    hits = sum(random.randrange(i) == 0 for _ in range(30000))
    print(f"i={i}: tần suất {hits / 30000:.3f}  (kỳ vọng {1 / i:.3f})")

i=1: tần suất 1.000  (kỳ vọng 1.000)
i=2: tần suất 0.496  (kỳ vọng 0.500)
i=3: tần suất 0.335  (kỳ vọng 0.333)
i=5: tần suất 0.201  (kỳ vọng 0.200)


### 3.4. Phân số `Fraction`

`Fraction(1, 3)` là phân số **chính xác** (không sai số làm tròn như `0.333...`). Ta sẽ dùng để chứng minh xác suất bằng đúng `1/n`.

In [5]:
print(Fraction(1, 2) * Fraction(2, 3) * Fraction(3, 4))      # 1/4
print(Fraction(1, 3) + Fraction(1, 6))                       # 1/2
print(1 - Fraction(1, 5))                                    # 4/5

1/4
1/2
4/5


### 3.5. Test ngẫu nhiên: tính chất + thống kê

Ta **không** so sánh một dãy ngẫu nhiên cụ thể. Thay vào đó: (a) giá trị trả về luôn **thuộc danh sách**; (b) với `trials` lần, đếm tần suất từng nút; mỗi nút phải có số lần trong khoảng `trials/n ± 5σ` với `σ = sqrt(trials · p · (1-p))`, `p = 1/n`. Với seed cố định kết quả lặp lại 100%.

In [6]:
def check_uniform_counts(counts, keys, trials):
    """Mỗi key trong keys phải xuất hiện ~ trials/k lần (sai lệch < 5 sigma). Trả về độ lệch lớn nhất."""
    k = len(keys)
    p = 1 / k
    sigma = (trials * p * (1 - p)) ** 0.5 if k > 1 else 1
    assert set(counts) <= set(keys), "có giá trị lạ"
    worst = max(abs(counts.get(key, 0) - trials * p) / sigma for key in keys)
    assert worst < 5, worst
    return worst


# thử trên bộ tung xúc xắc "chuẩn"
random.seed(0)
cnt = Counter(random.randrange(6) for _ in range(60000))
print(sorted(cnt.items()), "lệch", round(check_uniform_counts(cnt, range(6), 60000), 2), "sigma")

[(0, 9978), (1, 10136), (2, 9994), (3, 9825), (4, 10091), (5, 9976)] lệch 1.92 sigma


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Bạn ngồi xem một **đoàn tàu đi qua** từng toa, **không biết có bao nhiêu toa**, và phải nói to "toa tôi chọn" sao cho mỗi toa có cơ hội bằng nhau, mà chỉ được **nhớ một toa**.

Thử tính tay với đoàn **3 toa** (A, B, C):

- Toa 1 (A): chưa có gì, chọn A (xác suất `1/1`).
- Toa 2 (B): "tung đồng xu công bằng": với xác suất `1/2` đổi sang B, `1/2` giữ A. Giờ A và B đều có xác suất `1/2`.
- Toa 3 (C): đổi sang C với xác suất `1/3`; nếu **không** đổi (xác suất `2/3`) thì giữ nguyên lựa chọn cũ.
  - P(C) = `1/3`.
  - P(A) = P(A sau toa 2) × P(không đổi) = `1/2 × 2/3 = 1/3`.
  - P(B) = `1/2 × 2/3 = 1/3`.

Cả ba đều `1/3`! Quy luật: **ở toa thứ `i` đổi với xác suất `1/i`**.

## 5. Cách 1 — Brute Force / baseline: chép ra mảng

**Ý tưởng:** trong `__init__`, duyệt một lượt chép mọi giá trị vào `self.vals`. `getRandom()` trả `random.choice(self.vals)` — chọn chỉ số ngẫu nhiên đều.

Đúng và rất nhanh khi gọi (`O(1)`), nhưng tốn **`O(n)` bộ nhớ** và **cần đọc hết danh sách trước**, trái với follow up.

In [7]:
class SolutionArray:
    """Cách 1: chép ra mảng. __init__ O(n), getRandom O(1), Space O(n)."""

    def __init__(self, head):
        self.vals = []
        while head:
            self.vals.append(head.val)
            head = head.next

    def getRandom(self) -> int:
        return random.choice(self.vals)


random.seed(1)
print(run_ops(SolutionArray, ops, args))
assert set(run_ops(SolutionArray, ops, args)[1:]) <= {1, 2, 3}

[None, 1, 3, 1, 2, 1]


**Độ phức tạp:** khởi tạo `O(n)`, mỗi lần `getRandom` `O(1)`, bộ nhớ `O(n)`.

**Nút thắt:** phải **lưu toàn bộ** `n` giá trị. Nếu danh sách khổng lồ (hay là một luồng dữ liệu không biết độ dài) thì không thể. Ta muốn: **một lượt duyệt, bộ nhớ `O(1)`**.

## 6. Ý tưởng thuật toán / bước đột phá

Có hai hướng bỏ mảng phụ:

- **(a) Đếm rồi nhảy:** mỗi lần gọi, đếm độ dài `n`, chọn chỉ số `k = randrange(n)` rồi đi tới nút `k`. Bộ nhớ `O(1)`, nhưng **hai lượt duyệt** mỗi lần và **phải biết độ dài** (phải đi hết một lượt).
- **(b) Reservoir Sampling (lấy mẫu bể chứa):** chỉ **một lượt**, giữ **một** nút "đáp án hiện tại" `ans`. Ở nút thứ `i` (đếm từ `1`) **thay `ans` bằng nút này với xác suất `1/i`**.

**Vì sao (b) cho xác suất đều?** Nút thứ `j` (với `j <= n`) là đáp án cuối khi:

1. Nó **được chọn** ở bước `j`: xác suất `1/j`.
2. **Không bị thay** ở các bước `j+1, ..., n`: xác suất `(1 - 1/(j+1)) · (1 - 1/(j+2)) · ... · (1 - 1/n) = (j/(j+1)) · ((j+1)/(j+2)) · ... · ((n-1)/n) = j/n` (các tử/mẫu triệt tiêu).

Nhân lại: `1/j · j/n = 1/n`. Với **mọi** `j`, nên đều và **không cần biết `n` từ trước**.

In [8]:
def prob_final(j, n):
    """Xác suất CHÍNH XÁC nút thứ j là đáp án cuối khi danh sách có n nút."""
    p = Fraction(1, j)                       # được chọn ở bước j
    for t in range(j + 1, n + 1):
        p *= 1 - Fraction(1, t)              # không bị thay ở bước t
    return p


for n in (1, 2, 3, 7, 50):
    ps = [prob_final(j, n) for j in range(1, n + 1)]
    assert all(p == Fraction(1, n) for p in ps)
    print(f"n={n:>2}: mọi nút đều có xác suất {ps[0]}")

n= 1: mọi nút đều có xác suất 1
n= 2: mọi nút đều có xác suất 1/2
n= 3: mọi nút đều có xác suất 1/3
n= 7: mọi nút đều có xác suất 1/7
n=50: mọi nút đều có xác suất 1/50


## 7. Các cách tiếp theo

Tổng cộng 4 cách: Cách 1 (chép mảng, mục 5), Cách 2 (đếm độ dài rồi nhảy), Cách 3 (Reservoir Sampling — **chính thức**), Cách 4 (gán khóa ngẫu nhiên, lấy nhỏ nhất).

### Cách 2 — Đếm độ dài mỗi lần, rồi nhảy tới nút ngẫu nhiên

Giải thích từng đoạn:

1. **Lượt 1:** duyệt đếm `n`.
2. Chọn `k = random.randrange(n)` (chỉ số `0..n-1`, đều).
3. **Lượt 2:** bước `k` lần từ `head` rồi trả `node.val`.

Mỗi nút trúng với xác suất `1/n`. Không lưu mảng nên bộ nhớ `O(1)`, nhưng **mỗi lần gọi** tốn tới `2n` bước.

In [9]:
class SolutionCount:
    """Cách 2: mỗi lần gọi đếm n rồi nhảy. getRandom O(n), Space O(1)."""

    def __init__(self, head):
        self.head = head

    def getRandom(self) -> int:
        n = 0
        node = self.head
        while node:                      # lượt 1: đếm
            n += 1
            node = node.next
        k = random.randrange(n)          # chỉ số 0..n-1
        node = self.head
        for _ in range(k):               # lượt 2: nhảy k bước
            node = node.next
        return node.val


random.seed(2)
res = run_ops(SolutionCount, ops, args)
print(res)
assert set(res[1:]) <= {1, 2, 3}

[None, 1, 1, 1, 2, 1]


**Code nộp LeetCode cho Cách 2:**

```python
import random

class Solution:
    def __init__(self, head: Optional[ListNode]):
        self.head = head

    def getRandom(self) -> int:
        n = 0
        node = self.head
        while node:
            n += 1
            node = node.next
        k = random.randrange(n)
        node = self.head
        for _ in range(k):
            node = node.next
        return node.val
```

**Độ phức tạp:** `getRandom` `O(n)` (hai lượt), bộ nhớ `O(1)`. Có thể cải thiện bằng cách đếm `n` **một lần** trong `__init__` (khi danh sách không đổi) — nhưng khi không biết độ dài/luồng dữ liệu thì không được.

---

### Cách 3 — Reservoir Sampling (một lượt, `O(1)` bộ nhớ) — chính thức

Giải thích từng đoạn:

1. `ans = None`, `i = 0` (số nút đã đi qua).
2. Duyệt mỗi nút: `i += 1` (nút này là nút thứ `i`).
3. Tung "xúc xắc `i` mặt": `if random.randrange(i) == 0: ans = node.val` — xác suất đúng `1/i`.
4. Hết danh sách, trả `ans`.

Ở nút đầu `i = 1`: `randrange(1)` luôn `0` ⇒ chắc chắn nhận, nên `ans` không bao giờ `None` nếu danh sách không rỗng.

In [10]:
class SolutionReservoir:
    """Cách 3 (demo): Reservoir Sampling k=1."""

    def __init__(self, head):
        self.head = head

    def getRandom(self) -> int:
        ans = None
        i = 0
        node = self.head
        while node:
            i += 1                                # nút thứ i
            if random.randrange(i) == 0:          # xác suất 1/i
                ans = node.val
            node = node.next
        return ans


random.seed(3)
res = run_ops(SolutionReservoir, ops, args)
print(res)
assert set(res[1:]) <= {1, 2, 3}

[None, 2, 2, 1, 2, 1]


**Code nộp LeetCode cho Cách 3:**

```python
import random

class Solution:
    def __init__(self, head: Optional[ListNode]):
        self.head = head

    def getRandom(self) -> int:
        ans = None
        i = 0
        node = self.head
        while node:
            i += 1
            if random.randrange(i) == 0:
                ans = node.val
            node = node.next
        return ans
```

**Độ phức tạp:** `getRandom` `O(n)` (một lượt), bộ nhớ `O(1)`, **không cần biết `n`**. Đây là lời giải cho follow up.

---

### Cách 4 — Gán khóa ngẫu nhiên cho mỗi nút, lấy nút có khóa nhỏ nhất

**Ý tưởng khác:** mỗi nút bốc một số thực ngẫu nhiên độc lập `u ∈ [0, 1)`. Nút có `u` **nhỏ nhất** là đáp án. Vì các `u` là **độc lập, cùng phân phối**, mỗi nút có cơ hội như nhau để là nhỏ nhất (đối xứng ⇒ xác suất `1/n`, khả năng hòa gần như bằng `0`).

Cũng một lượt, `O(1)` bộ nhớ (chỉ giữ khóa nhỏ nhất và giá trị tương ứng). Điểm hay: dễ tổng quát thành "chọn `k` nút" (giữ `k` khóa nhỏ nhất bằng heap), xem mục 15.

In [11]:
class SolutionMinKey:
    """Cách 4: mỗi nút bốc khóa ngẫu nhiên, giữ nút có khóa nhỏ nhất."""

    def __init__(self, head):
        self.head = head

    def getRandom(self) -> int:
        best_key, ans = 2.0, None            # 2.0 > mọi khóa trong [0,1)
        node = self.head
        while node:
            key = random.random()
            if key < best_key:
                best_key, ans = key, node.val
            node = node.next
        return ans


random.seed(4)
res = run_ops(SolutionMinKey, ops, args)
print(res)
assert set(res[1:]) <= {1, 2, 3}

[None, 2, 2, 3, 1, 2]


**Code nộp LeetCode cho Cách 4:**

```python
import random

class Solution:
    def __init__(self, head: Optional[ListNode]):
        self.head = head

    def getRandom(self) -> int:
        best_key, ans = 2.0, None
        node = self.head
        while node:
            key = random.random()
            if key < best_key:
                best_key, ans = key, node.val
            node = node.next
        return ans
```

**Độ phức tạp:** `getRandom` `O(n)`, bộ nhớ `O(1)`. Gọi `random.random()` ở mọi nút (Cách 3 cũng gọi một lần mỗi nút), nên chi phí ngang nhau.

## 8. TRACE từng bước

Trace Cách 3 trên `[10, 20, 30, 40]`: mỗi bước ta in `i`, `val`, kết quả `randrange(i)`, quyết định thay hay không, và `ans` mới.

| Nút | `i` | `r = randrange(i)` | `r == 0`? | `ans` sau bước |
|---|---|---|---|---|
| 10 | 1 | 0 (luôn) | có | 10 |
| 20 | 2 | 0 hoặc 1 | tùy | 20 nếu `r=0` |
| 30 | 3 | 0, 1, 2 | tùy | 30 nếu `r=0` |
| 40 | 4 | 0..3 | tùy | 40 nếu `r=0` |

Hàm sau chạy thật với seed và in từng dòng.

In [12]:
def trace_reservoir(vals, seed):
    random.seed(seed)
    node = build_list(vals)
    ans, i = None, 0
    print(f"seed={seed}, danh sách {vals}")
    print(f"{'i':<3}{'val':<6}{'r':<4}{'thay?':<7}{'ans':<5}")
    while node:
        i += 1
        r = random.randrange(i)
        replaced = (r == 0)
        if replaced:
            ans = node.val
        print(f"{i:<3}{node.val:<6}{r:<4}{str(replaced):<7}{ans}")
        node = node.next
    print("=> trả về", ans, "\n")
    return ans


trace_reservoir([10, 20, 30, 40], seed=0)
trace_reservoir([10, 20, 30, 40], seed=1)
trace_reservoir([10, 20, 30, 40], seed=2)

seed=0, danh sách [10, 20, 30, 40]
i  val   r   thay?  ans  
1  10    0   True   10
2  20    1   False  10
3  30    0   True   30
4  40    2   False  30
=> trả về 30 

seed=1, danh sách [10, 20, 30, 40]
i  val   r   thay?  ans  
1  10    0   True   10
2  20    0   True   20
3  30    1   False  20
4  40    0   True   40
=> trả về 40 

seed=2, danh sách [10, 20, 30, 40]
i  val   r   thay?  ans  
1  10    0   True   10
2  20    0   True   20
3  30    0   True   30
4  40    2   False  30
=> trả về 30 



30

Mỗi seed cho một kết quả khác nhau, nhưng **không** seed nào "đúng hơn": đúng là xác suất, không phải một giá trị. Muốn thấy rõ tính đều **chính xác** (không phải ước lượng), ta **liệt kê mọi tổ hợp** kết quả tung `r_1, r_2, r_3, r_4` (`r_i` ∈ `[0, i)`): có `1·2·3·4 = 24` tổ hợp, mỗi tổ hợp xác suất như nhau. Đếm xem mỗi nút là đáp án cuối bao nhiêu lần:

In [13]:
def reservoir_with_choices(vals, choices):
    """Reservoir với 'kết quả xúc xắc' cho trước: choices[i-1] là r của bước i."""
    ans = None
    for i, v in enumerate(vals, start=1):
        if choices[i - 1] == 0:
            ans = v
    return ans


def exact_distribution(vals):
    n = len(vals)
    ranges = [range(i) for i in range(1, n + 1)]      # r_i in [0, i)
    cnt = Counter()
    total = 0
    for choices in product(*ranges):
        cnt[reservoir_with_choices(vals, choices)] += 1
        total += 1
    return cnt, total


cnt, total = exact_distribution([10, 20, 30, 40])
print("tổng tổ hợp:", total)
print({k: f"{v}/{total}" for k, v in sorted(cnt.items())})
assert set(cnt.values()) == {total // 4}      # mỗi nút đúng 6/24

tổng tổ hợp: 24
{10: '6/24', 20: '6/24', 30: '6/24', 40: '6/24'}


Mỗi nút xuất hiện **đúng 6 trong 24** tổ hợp ⇒ xác suất **chính xác `1/4`**. Đây là chứng minh bằng vét cạn cho `n = 4`; mục 6 đã chứng minh tổng quát.

## 9. Code nộp LeetCode

Phiên bản cuối (Cách 3, trả lời được follow up), đúng tên lớp/method/signature:

In [14]:
class Solution:

    def __init__(self, head: Optional[ListNode]):
        self.head = head

    def getRandom(self) -> int:
        ans = None
        i = 0
        node = self.head
        while node:
            i += 1
            if random.randrange(i) == 0:
                ans = node.val
            node = node.next
        return ans


random.seed(10)
print(run_ops(Solution, ops, args))

[None, 1, 2, 1, 2, 3]


## 10. Vì sao thuật toán đúng

- **Luôn có đáp án:** ở nút đầu `i = 1`, `randrange(1) == 0` luôn đúng nên `ans` được gán; các bước sau chỉ ghi đè bằng giá trị hợp lệ.
- **Xác suất đều:** đã chứng minh ở mục 6: nút `j` là kết quả cuối với xác suất `1/j · ∏_{t=j+1}^{n} (1 - 1/t) = 1/j · j/n = 1/n`. Mục 6 và 8 đều kiểm bằng `Fraction` và vét cạn.
- **Độc lập giữa các lần gọi:** mỗi lần `getRandom` dùng các lần `randrange` mới, **không lưu trạng thái** nào trong `self` ⇒ các lần gọi không ảnh hưởng nhau.
- **Giá trị trùng:** ta chọn theo **nút**, nên giá trị xuất hiện `c` lần có xác suất `c/n` — đúng yêu cầu "mọi **nút**" đều như nhau.

## 11. Phân tích Time / Space Complexity

| Cách | `__init__` | `getRandom` | Space |
|---|---|---|---|
| 1. Chép mảng | `O(n)` | `O(1)` | `O(n)` |
| 2. Đếm + nhảy | `O(1)` | `O(n)` (2 lượt) | `O(1)` |
| 3. Reservoir | `O(1)` | `O(n)` (1 lượt) | `O(1)` |
| 4. Khóa nhỏ nhất | `O(1)` | `O(n)` | `O(1)` |

Với `n = 10^4` và `10^4` lần gọi, các cách `O(n)` mỗi lần cần cỡ `10^8` bước — vẫn chấp nhận được trên LeetCode (đề đặt giới hạn nhỏ như vậy), nhưng Cách 1 nhanh nhất khi gọi nhiều. Cách 3 là **cách duy nhất vừa không cần biết `n`, vừa không lưu mảng**: đó là ý nghĩa của follow up.

## 12. Bộ test

Chiến lược (bài ngẫu nhiên, test bằng **tính chất + thống kê có seed**):

1. **Ví dụ đề:** mọi giá trị trả về thuộc `{1, 2, 3}`.
2. **Edge case:** một nút (luôn trả giá trị đó), hai nút, giá trị âm, giá trị **trùng** (`[1, 1, 2]` ⇒ `1` xuất hiện `2/3`), `n = 10^4`.
3. **Thống kê:** mỗi nút xuất hiện ≈ `trials/n` lần (≤ 5σ), cho cả 4 cách.
4. **Kiểm tra chính xác bằng vét cạn** (mục 8) cho nhiều độ dài.
5. **Đối chiếu hai cách** (Reservoir vs chép mảng): phân phối thực nghiệm gần nhau.

In [15]:
IMPLS = [SolutionArray, SolutionCount, SolutionReservoir, SolutionMinKey, Solution]

# 1) Ví dụ đề
for cls in IMPLS:
    random.seed(7)
    r = run_ops(cls, ops, args)
    assert r[0] is None and len(r) == 6 and set(r[1:]) <= {1, 2, 3}
print("Ví dụ đề: OK")

# 2) Edge cases
for cls in IMPLS:
    random.seed(8)
    assert {cls(build_list([5])).getRandom() for _ in range(50)} == {5}            # một nút
    assert {cls(build_list([-3, -4])).getRandom() for _ in range(200)} == {-3, -4} # hai nút, số âm
    assert {cls(build_list([0])).getRandom() for _ in range(10)} == {0}            # giá trị 0
print("Edge cases: OK")

Ví dụ đề: OK
Edge cases: OK


In [16]:
# 2b) Giá trị trùng: xác suất theo NÚT, nên giá trị 1 chiếm 2/3
for cls in IMPLS:
    random.seed(9)
    s = cls(build_list([1, 1, 2]))
    cnt = Counter(s.getRandom() for _ in range(30000))
    frac1 = cnt[1] / 30000
    assert abs(frac1 - 2 / 3) < 0.02, (cls.__name__, frac1)
    print(f"{cls.__name__:<18} P(1) ≈ {frac1:.3f} (kỳ vọng 0.667)")

SolutionArray      P(1) ≈ 0.663 (kỳ vọng 0.667)
SolutionCount      P(1) ≈ 0.663 (kỳ vọng 0.667)
SolutionReservoir  P(1) ≈ 0.664 (kỳ vọng 0.667)


SolutionMinKey     P(1) ≈ 0.673 (kỳ vọng 0.667)


Solution           P(1) ≈ 0.664 (kỳ vọng 0.667)


In [17]:
# 3) Thống kê: mỗi NÚT (dùng giá trị đôi một khác nhau để đếm theo nút)
for n, trials in [(3, 30000), (5, 50000), (10, 100000)]:
    vals = list(range(100, 100 + n))
    for cls in IMPLS:
        random.seed(1000 + n)
        s = cls(build_list(vals))
        counts = Counter(s.getRandom() for _ in range(trials))
        worst = check_uniform_counts(counts, vals, trials)
    print(f"n={n:>2}, {trials} lần: cả {len(IMPLS)} cách đều đều (lệch tối đa gần nhất {worst:.2f} sigma)")

n= 3, 30000 lần: cả 5 cách đều đều (lệch tối đa gần nhất 0.55 sigma)


n= 5, 50000 lần: cả 5 cách đều đều (lệch tối đa gần nhất 1.91 sigma)


n=10, 100000 lần: cả 5 cách đều đều (lệch tối đa gần nhất 1.74 sigma)


In [18]:
# 4) Vét cạn chính xác cho nhiều độ dài (không cần ngẫu nhiên)
for n in range(1, 8):
    vals = list(range(n))
    cnt, total = exact_distribution(vals)
    assert total == __import__("math").factorial(n)
    assert all(cnt[v] == total // n for v in vals), (n, cnt)
print("Vét cạn: n = 1..7 đều chính xác 1/n")

# 5) Đối chiếu Reservoir với chép mảng trên danh sách dài hơn
vals = list(range(20))
random.seed(31337)
a = Counter(SolutionReservoir(build_list(vals)).getRandom() for _ in range(40000))
b = Counter(SolutionArray(build_list(vals)).getRandom() for _ in range(40000))
check_uniform_counts(a, vals, 40000); check_uniform_counts(b, vals, 40000)
print("n=20: Reservoir và Array đều đều")

Vét cạn: n = 1..7 đều chính xác 1/n


n=20: Reservoir và Array đều đều


In [19]:
# 6) Danh sách dài tối đa theo ràng buộc: chỉ cần chạy được và đáp án hợp lệ
big = list(range(10000))
random.seed(5)
s = Solution(build_list(big))
t = time.perf_counter()
vals_out = [s.getRandom() for _ in range(300)]
print(f"n=10^4, 300 lần gọi: {time.perf_counter() - t:.3f}s")
assert all(0 <= v < 10000 for v in vals_out)
# vị trí trung bình nên gần 5000 (sai số ~ 2887/sqrt(300) ~ 167, chặn ở 5 sigma)
mean = sum(vals_out) / len(vals_out)
assert abs(mean - 4999.5) < 5 * 2887 / 300 ** 0.5, mean
print("trung bình ≈", round(mean))

n=10^4, 300 lần gọi: 0.773s
trung bình ≈ 4550


## 13. Các lỗi người mới hay gặp

### Lỗi 1: thay với xác suất `1/2` thay vì `1/i`

Nghĩ "mỗi nút tung đồng xu" thì thiên vị các nút **cuối**: nút `n` có xác suất `1/2`, nút `n-1` chỉ `1/4`, ...

In [20]:
class BadHalf:
    def __init__(self, head): self.head = head
    def getRandom(self):
        ans, node, first = None, self.head, True
        while node:
            if first or random.random() < 0.5:     # SAI: luôn 1/2
                ans = node.val
            first = False
            node = node.next
        return ans

random.seed(0)
cnt = Counter(BadHalf(build_list([1, 2, 3])).getRandom() for _ in range(40000))
print({k: round(v / 40000, 3) for k, v in sorted(cnt.items())}, "(đúng phải ~0.333 mỗi giá trị)")

{1: 0.249, 2: 0.249, 3: 0.502} (đúng phải ~0.333 mỗi giá trị)


Kỳ vọng sai: `1 → 1/4`, `2 → 1/4`, `3 → 1/2`. **Sửa:** xác suất phải là `1/i` với `i` là thứ tự nút (`randrange(i) == 0`).

### Lỗi 2: sai biên của `randrange` / `randint`

`random.randint(0, n)` **bao gồm cả `n`** nên chỉ số có thể bằng `n` — nằm ngoài danh sách (Cách 2). Hoặc dùng `randint(0, i)` khi `i` đã đếm từ `1` ⇒ xác suất `1/(i+1)` chứ không phải `1/i`.

In [21]:
class BadBoundary:
    def __init__(self, head): self.head = head
    def getRandom(self):
        n, node = 0, self.head
        while node:
            n += 1; node = node.next
        k = random.randint(0, n)          # SAI: có thể ra n (bao gồm cả n)
        node = self.head
        for _ in range(k):
            node = node.next
        return node.val                   # k == n -> node là None -> AttributeError

random.seed(1)
s = BadBoundary(build_list([1, 2, 3]))
try:
    for _ in range(50):
        s.getRandom()
except AttributeError as e:
    print("AttributeError:", e)

AttributeError: 'NoneType' object has no attribute 'val'


**Sửa:** `random.randrange(n)` (loại trừ `n`) hoặc `random.randint(0, n - 1)`.

### Lỗi 3: bắt đầu đếm `i` từ `0`

Nếu `i = 0` rồi gọi ngay `random.randrange(i)` ở nút đầu ⇒ `ValueError` (khoảng rỗng). Phải tăng `i` **trước** khi tung (nút đầu là `i = 1`).

### Lỗi 4: nhầm "giá trị" với "nút" khi có trùng

Nếu "khử trùng" giá trị (dùng `set`) rồi chọn đều thì sai: `[1, 1, 2]` phải ra `1` với xác suất `2/3`, không phải `1/2`. Đề nói **mọi nút** đều như nhau.

### Lỗi 5: test ngẫu nhiên bằng so sánh cứng

Viết `assert solution.getRandom() == 1` rồi "chạy được" vì tình cờ seed. Đúng: test **tính chất** (thuộc danh sách) và **thống kê** (mục 12), hoặc **vét cạn** các kết quả tung xúc xắc.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **"Chọn đều một phần tử từ luồng/danh sách không biết độ dài, `O(1)` bộ nhớ"** ⇒ **Reservoir Sampling**.
- Công thức vàng: ở phần tử thứ `i`, thay đáp án với xác suất `1/i` (`random.randrange(i) == 0`). Nhớ chứng minh bằng "tích triệt tiêu" `j/n`.
- **Chọn `k` phần tử đều** ⇒ giữ `k` ô; phần tử thứ `i > k` thay một ô ngẫu nhiên với xác suất `k/i`.
- **Có thể biết `n`** (gọi nhiều lần) ⇒ chép mảng/đếm trước cho nhanh; **không biết `n`** ⇒ reservoir.
- Bài ngẫu nhiên: test bằng **seed + thống kê (≤ 5σ)** và, khi không gian nhỏ, **vét cạn** các kết quả tung xúc xắc.

## 15. Biến thể của bài

1. **LeetCode 398 — Random Pick Index:** mảng có thể trùng, chọn đều **chỉ số** của phần tử bằng `target`. Reservoir chỉ đếm các vị trí có `nums[i] == target`.
2. **Chọn `k` nút đều (Algorithm R):** giữ `k` ô; nút thứ `i > k` thay một ô ngẫu nhiên với xác suất `k/i`.
3. **Reservoir có trọng số:** gán khóa `u^(1/w)` (hoặc `-log(u)/w`) rồi giữ khóa lớn/nhỏ nhất — mở rộng của Cách 4.
4. **LeetCode 384 — Shuffle an Array:** trộn đều toàn bộ mảng (Fisher–Yates).

Cài thử biến thể 1 và 2:

In [22]:
def random_pick_index(nums, target):
    """Reservoir: chọn đều một chỉ số i có nums[i] == target (một lượt, O(1) bộ nhớ)."""
    ans, seen = -1, 0
    for i, x in enumerate(nums):
        if x == target:
            seen += 1
            if random.randrange(seen) == 0:
                ans = i
    return ans


def reservoir_k(stream, k):
    """Algorithm R: k phần tử đều, không lặp (theo vị trí)."""
    res = []
    for i, x in enumerate(stream, start=1):
        if i <= k:
            res.append(x)
        else:
            j = random.randrange(i)            # 0..i-1
            if j < k:                          # xác suất k/i
                res[j] = x
    return res


random.seed(12)
c = Counter(random_pick_index([1, 3, 3, 2, 3], 3) for _ in range(30000))
print(sorted(c.items()))
check_uniform_counts(c, [1, 2, 4], 30000)

c2 = Counter()
for _ in range(20000):
    for x in reservoir_k(range(5), 2):
        c2[x] += 1
print(sorted(c2.items()), "(mỗi số ~ 20000 * 2/5 = 8000)")
assert all(abs(v - 8000) < 5 * (20000 * 0.4 * 0.6) ** 0.5 for v in c2.values())

[(1, 10057), (2, 10006), (4, 9937)]
[(0, 8110), (1, 7982), (2, 8041), (3, 7923), (4, 7944)] (mỗi số ~ 20000 * 2/5 = 8000)


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time `getRandom` | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Chép mảng | lưu hết, `choice` | `O(1)` (sau `O(n)` khởi tạo) | `O(n)` | Gọi **rất nhiều** lần, danh sách không đổi, vừa bộ nhớ |
| 2. Đếm + nhảy | đếm `n`, chọn chỉ số, đi tới đó | `O(n)` (2 lượt) | `O(1)` | Biết cách làm "thuần chỉ số", ít bộ nhớ |
| 3. Reservoir | thay đáp án với xác suất `1/i` | `O(n)` (1 lượt) | `O(1)` | **Không biết độ dài/luồng dữ liệu**; lời giải cho follow up |
| 4. Khóa nhỏ nhất | khóa ngẫu nhiên, lấy nhỏ nhất | `O(n)` | `O(1)` | Dễ mở rộng sang `k` phần tử / có trọng số |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code. Với danh sách `[7]` (một nút), `getRandom()` (Cách 3) trả gì **luôn luôn**? Và `random.randrange(1)` trả gì?

**Bài 2 (tính tay).** Với danh sách 3 nút `A, B, C`, giả sử các lần tung là `r_1 = 0, r_2 = 1, r_3 = 0`. Đáp án cuối là nút nào? Nếu `r_3 = 2` thì sao? Hãy tính tay xác suất nút `B` là đáp án cuối khi `n = 3` (đáp án: `1/3`) bằng công thức mục 6.

**Bài 3 (điền TODO).** Hoàn thành `reservoir_todo(vals)` trả một giá trị theo Reservoir Sampling.

In [23]:
def reservoir_todo(vals):
    # TODO 1: ans = None
    # TODO 2: duyệt vals với chỉ số i bắt đầu từ 1 (enumerate(vals, start=1))
    # TODO 3: nếu random.randrange(i) == 0 thì ans = giá trị hiện tại
    # TODO 4: trả ans
    return None

# KHÔNG gọi reservoir_todo ở đây để Run All không lỗi (TODO chưa điền).

**Bài 4 (tự viết lại).** Viết `class SolutionKnownLength` (đếm độ dài **một lần** trong `__init__`, `getRandom` đi `randrange(n)` bước) **mà không nhìn lại** mục 7. Kiểm tra bằng `check_uniform_counts`.

**Bài 5 (biến thể / mini challenge).** Viết `reservoir_weighted(items, weights)` với `items` là danh sách giá trị, `weights` là trọng số dương nguyên: đầu ra phải chọn `items[i]` với xác suất `weights[i] / sum(weights)`. Gợi ý: duy trì `total` (tổng trọng số đã thấy), ở phần tử thứ `i` đổi đáp án với xác suất `weights[i] / total` (dùng `random.randrange(total) < weights[i]`). Kiểm tra với `items = ['a','b','c']`, `weights = [1, 2, 7]`.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: luôn `7`; `randrange(1)` luôn `0`. Bài 3: `for i, v in enumerate(vals, start=1):`.
- **Mức 2 (vừa):** Bài 2: `r_1 = 0` ⇒ `ans = A`; `r_2 = 1 ≠ 0` ⇒ giữ `A`; `r_3 = 0` ⇒ `ans = C`. Với `r_3 = 2` ⇒ giữ `A`. Bài 4: đếm `n` rồi lưu `self.n`; `randrange(self.n)`.
- **Mức 3 (sâu):** Bài 5: chứng minh tương tự mục 6: xác suất phần tử `j` là cuối = `w_j/S_j · ∏ (1 - w_t/S_t) = w_j / S_n`, vì `1 - w_t/S_t = S_{t-1}/S_t` rồi triệt tiêu (với `S_t` là tổng trọng số `t` phần tử đầu).

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [24]:
# Bài 1
random.seed(0)
assert {Solution(build_list([7])).getRandom() for _ in range(30)} == {7}
assert {random.randrange(1) for _ in range(30)} == {0}

# Bài 2
assert reservoir_with_choices(["A", "B", "C"], (0, 1, 0)) == "C"
assert reservoir_with_choices(["A", "B", "C"], (0, 1, 2)) == "A"
assert reservoir_with_choices(["A", "B", "C"], (0, 0, 1)) == "B"
assert prob_final(2, 3) == Fraction(1, 3)


# Bài 3
def reservoir_todo_ans(vals):
    ans = None
    for i, v in enumerate(vals, start=1):
        if random.randrange(i) == 0:
            ans = v
    return ans


random.seed(21)
cnt3 = Counter(reservoir_todo_ans([1, 2, 3, 4]) for _ in range(40000))
check_uniform_counts(cnt3, [1, 2, 3, 4], 40000)


# Bài 4
class SolutionKnownLength:
    def __init__(self, head):
        self.head = head
        self.n = 0
        node = head
        while node:
            self.n += 1
            node = node.next

    def getRandom(self):
        node = self.head
        for _ in range(random.randrange(self.n)):
            node = node.next
        return node.val


random.seed(22)
sk = SolutionKnownLength(build_list([5, 6, 7, 8]))
cnt4 = Counter(sk.getRandom() for _ in range(40000))
check_uniform_counts(cnt4, [5, 6, 7, 8], 40000)


# Bài 5
def reservoir_weighted(items, weights):
    ans, total = None, 0
    for x, w in zip(items, weights):
        total += w
        if random.randrange(total) < w:          # xác suất w / total
            ans = x
    return ans


random.seed(23)
cnt5 = Counter(reservoir_weighted(['a', 'b', 'c'], [1, 2, 7]) for _ in range(50000))
for key, w in zip('abc', [1, 2, 7]):
    p = w / 10
    assert abs(cnt5[key] / 50000 - p) < 5 * (p * (1 - p) / 50000) ** 0.5, (key, cnt5[key])
print("Tất cả đáp án đúng")

Tất cả đáp án đúng


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (Reservoir Sampling `k = 1`)

```python
import random

class Solution:
    def __init__(self, head: Optional[ListNode]):
        self.head = head

    def getRandom(self) -> int:
        ans = None
        i = 0
        node = self.head
        while node:
            i += 1
            if random.randrange(i) == 0:   # xác suất 1/i
                ans = node.val
            node = node.next
        return ans
```

**Checklist**

- [ ] `i` được tăng **trước** khi tung xúc xắc (nút đầu là `i = 1`)?
- [ ] Xác suất thay là **`1/i`** (`randrange(i) == 0`), không phải `1/2`?
- [ ] Không lưu trạng thái ngẫu nhiên giữa các lần gọi (mỗi lần `getRandom` chạy từ `head`)?
- [ ] Biên `randrange(n)` loại trừ `n`; không dùng `randint(0, n)` nhầm?
- [ ] Xác suất tính theo **nút** (giá trị trùng có xác suất lớn hơn)?
- [ ] Test bằng **thống kê có seed** (≤ 5σ) hoặc **vét cạn** các kết quả tung, không so cứng một dãy?
- [ ] Đã thử: danh sách một nút, hai nút, giá trị âm/0/trùng, `n = 10^4`?